# E16 — 이탈·복귀·종료 진단 (재학습 없음)
첨부 envs.py에 잔디 조기 종료 규칙은 없습니다. 정책·보상·종료 조건을 유지하고 원본 프레임마다 진단만 기록합니다. 내부 상태를 행동에 전달하지 않습니다. 접선 기반 방향 판정은 근사이며 영상과 대조하세요.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json, shutil, tempfile
ROOT=Path('/content/drive/MyDrive/261103')
RESULTS=ROOT/'261103_results'
if not (ROOT/'train.py').exists():
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run(['git','clone','--depth','1','--branch','261103','https://github.com/MilkySon/RL_carracing_competition.git',tmp+'/repo'],check=True)
        shutil.copytree(Path(tmp)/'repo'/'261103',ROOT,dirs_exist_ok=True)
RESULTS.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
os.environ['SDL_VIDEODRIVER']='dummy'
os.environ['SDL_AUDIODRIVER']='dummy'
print(ROOT,RESULTS)


Mounted at /content/drive
/content/drive/MyDrive/261103 /content/drive/MyDrive/261103/261103_results


In [2]:
subprocess.run([sys.executable,'-m','pip','install','-q','swig'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import torch
print('GPU:',torch.cuda.is_available())
def run(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a') as f:
        p=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='');f.write(line);f.flush()
        rc=p.wait()
    if rc:raise RuntimeError(f'Failed ({rc}): {log}')


GPU: True


## 1. 모델 경로
E15 먼저 실행. 기존 모델 경로는 실제 Drive 위치로 수정. 없는 이전 모델은 표시 후 건너뜁니다.

In [3]:
CHECKPOINTS={'E15':RESULTS/'E15'/'best.pt'}
for name in ['E10','E12','E14']:
    CHECKPOINTS[name]=Path('/content/drive/MyDrive/261101carracing_results')/name/'best.pt'
OUT=RESULTS/'E16';OUT.mkdir(parents=True,exist_ok=True)
assert CHECKPOINTS['E15'].exists(),'E15 완료 또는 경로 수정 필요'


## 2. 종료 조건 및 프레임 로그
이탈=모든 바퀴의 도로 접촉 없음. 도로 재접촉과 정방향·새 타일 방문 복귀를 구별합니다. 연속 역주행 집계는 원본 프레임 단위입니다.

In [4]:
for name,ckpt in CHECKPOINTS.items():
    if not ckpt.exists():print('SKIP missing:',name,ckpt);continue
    run([sys.executable,'audit_termination.py','--ckpt',ckpt,'--seeds','1000-1009','--out',OUT/name],OUT/f'{name}_console.log')
import pandas as pd
for p in sorted(OUT.glob('*/episodes.csv')):
    print(p.parent.name);display(pd.read_csv(p))


/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
{'seed': 1000, 'return_': 572.3549488054529, 'raw_frames': 1000, 'offroad_frames': 131, 'exits': 3, 'road_reentries': 3, 'forward_progress_recoveries': 0, 'reverse_frames': 723, 'max_reverse_streak': 723, 'terminated': False, 'truncated': True, 'end_reason': 'time_limit'}
{'seed': 1001, 'return_': 598.7179487179358, 'raw_frames': 1000, 'offroad_frames': 81, 'exits': 4, 'road_reentries': 4, 'forward_progress_recoveries': 4, 'reverse_frames': 214, 'max_reverse_streak': 175, 'terminated': False, 'truncated': True, 'end_reason': 'time_limit'}
{'seed': 1002, 'return_': 863.6363636363463, 'raw_frames': 1000, 'offroad_frames': 72, 'ex

,seed,return_,raw_frames,offroad_frames,exits,road_reentries,forward_progress_recoveries,reverse_frames,max_reverse_streak,terminated,truncated,end_reason
0,1000,896.587031,1000,15,2,2,2,0,0,False,True,time_limit
1,1001,910.800000,892,0,0,0,0,0,0,True,False,environment_termination
2,1002,922.000000,780,0,0,0,0,0,0,True,False,environment_termination
3,1003,890.000000,1000,7,1,1,1,0,0,False,True,time_limit
4,1004,896.644295,1000,2,1,1,1,0,0,False,True,time_limit
5,1005,904.300000,957,0,0,0,0,0,0,True,False,environment_termination
6,1006,857.142857,1000,135,2,2,2,43,43,False,True,time_limit
7,1007,912.600000,874,0,0,0,0,0,0,True,False,environment_termination
8,1008,871.518987,1000,18,1,1,1,0,0,False,True,time_limit
9,1009,870.370370,1000,19,1,1,1,0,0,False,True,time_limit


E12


,seed,return_,raw_frames,offroad_frames,exits,road_reentries,forward_progress_recoveries,reverse_frames,max_reverse_streak,terminated,truncated,end_reason
0,1000,869.283276,1000,50,2,2,1,3,3,False,True,time_limit
1,1001,348.717949,1000,79,3,3,2,392,364,False,True,time_limit
2,1002,532.727273,1000,45,3,3,2,447,445,False,True,time_limit
3,1003,903.800000,962,0,0,0,0,0,0,True,False,environment_termination
4,1004,876.510067,1000,9,2,2,2,0,0,False,True,time_limit
5,1005,771.165644,1000,44,2,2,2,25,25,False,True,time_limit
6,1006,911.500000,885,0,0,0,0,0,0,True,False,environment_termination
7,1007,900.900000,991,0,0,0,0,0,0,True,False,environment_termination
8,1008,901.500000,985,0,0,0,0,0,0,True,False,environment_termination
9,1009,913.500000,865,0,0,0,0,0,0,True,False,environment_termination


E14


,seed,return_,raw_frames,offroad_frames,exits,road_reentries,forward_progress_recoveries,reverse_frames,max_reverse_streak,terminated,truncated,end_reason
0,1000,828.327645,1000,25,1,1,1,23,23,False,True,time_limit
1,1001,358.333333,1000,172,2,2,1,411,368,False,True,time_limit
2,1002,906.500000,935,0,0,0,0,0,0,True,False,environment_termination
3,1003,766.666667,1000,40,2,2,2,22,22,False,True,time_limit
4,1004,901.200000,988,0,0,0,0,0,0,True,False,environment_termination
5,1005,801.840491,1000,21,1,1,1,0,0,False,True,time_limit
6,1006,910.900000,891,0,0,0,0,0,0,True,False,environment_termination
7,1007,900.500000,995,0,0,0,0,0,0,True,False,environment_termination
8,1008,849.367089,1000,0,0,0,0,0,0,False,True,time_limit
9,1009,912.100000,879,0,0,0,0,0,0,True,False,environment_termination


E15


,seed,return_,raw_frames,offroad_frames,exits,road_reentries,forward_progress_recoveries,reverse_frames,max_reverse_streak,terminated,truncated,end_reason
0,1000,572.354949,1000,131,3,3,0,723,723,False,True,time_limit
1,1001,598.717949,1000,81,4,4,4,214,175,False,True,time_limit
2,1002,863.636364,1000,72,4,4,3,91,91,False,True,time_limit
3,1003,783.333333,1000,52,3,3,3,32,32,False,True,time_limit
4,1004,886.577181,1000,3,1,1,1,0,0,False,True,time_limit
5,1005,34.844172,309,81,1,0,0,33,33,True,False,out_of_bounds_inferred
6,1006,915.700000,843,0,0,0,0,0,0,True,False,environment_termination
7,1007,272.168285,1000,425,3,3,0,261,182,False,True,time_limit
8,1008,687.974684,1000,166,2,2,2,37,37,False,True,time_limit
9,1009,918.600000,814,0,0,0,0,0,0,True,False,environment_termination


## 3. seed 1001 실패 시작점 영상

In [5]:
for name,ckpt in CHECKPOINTS.items():
    if ckpt.exists():run([sys.executable,'evaluate.py','--ckpt',ckpt,'--seeds','1001','--video',OUT/name/'videos'],OUT/f'{name}_video_console.log')


/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
seed 1001:    598.7
SCORE (mean over 1 tracks): 598.7 ± 0.0
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
seed 1001:    910.8
SCORE (mean over 1 tracks): 910.8 ± 0.0
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/la

## 판단 메모
frames.csv: 이탈 → 재접촉 → 정방향/새 타일 진행의 순서 확인. 종료 원인은 관측에 따른 분류이며 영상·원본 환경 코드와 대조.
즉시 종료 규칙이 없으므로 잔디 20스텝 허용을 무조건 추가하지 않습니다. 복귀 실패가 반복될 때 E17, 이후 필요 시 E18 검토.